# Real Estate Market Forecasting

This notebook introduces a forecasting workflow for Italian residential real-estate quotations. The objective is to move from statistical association to **out-of-sample prediction**, while explicitly preventing temporal leakage.

**Pipeline:** municipality-semester panel → annual target → lagged features → time-based train/test split → baseline → regularized model → evaluation → feature importance → limitations.

> Forecasting is predictive, not causal. OMI quotations are market indicators and should not be interpreted as transaction prices.

## 1. Methodological design

We forecast next-year municipality price growth using information available up to the current year. The split is chronological: earlier observations are used for training and the most recent observations are reserved for testing.

The main target is **next-year price growth**. Features include lagged price growth, lagged NTN growth, lagged population growth and municipality-level price/market characteristics. No random train/test split is used because it would leak future information into the training sample.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
QUOTATIONS_DIR = PROJECT_ROOT / 'data' / 'raw' / 'quotations'
TRANSACTIONS_DIR = PROJECT_ROOT / 'data' / 'raw' / 'transactions'
POPULATION_DIR = PROJECT_ROOT / 'data' / 'raw' / 'population'
for path in [QUOTATIONS_DIR, TRANSACTIONS_DIR, POPULATION_DIR]:
    assert path.exists(), f'Missing source directory: {path}'
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

## 2. Build the annual municipality panel

Annual price is the mean of the two available semester-level median OMI quotation estimates. NTN and population are annual measures. Keeping one observation per municipality-year avoids duplicating annual variables across semesters.

In [ ]:
quotation_parts = []
for path in sorted(QUOTATIONS_DIR.glob('omi_quotations_*.csv')):
    match = re.search(r'_(\d{4})_(S[12])$', path.stem)
    if not match:
        continue
    year, semester = int(match.group(1)), match.group(2)
    df = pd.read_csv(path, sep=';', low_memory=False)
    df.columns = [str(c).replace('\ufeff', '').strip() for c in df.columns]
    required = ['Comune_ISTAT', 'Descr_Tipologia', 'Compr_min', 'Compr_max', 'Regione']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f'{path.name}: missing columns {missing}')
    df = df[required].copy()
    df['year'], df['semester'] = year, semester
    quotation_parts.append(df)
omi = pd.concat(quotation_parts, ignore_index=True)
omi['municipality_code'] = omi['Comune_ISTAT'].astype('string').str.strip()
omi['Compr_min'] = pd.to_numeric(omi['Compr_min'], errors='coerce')
omi['Compr_max'] = pd.to_numeric(omi['Compr_max'], errors='coerce')
omi['price_m2'] = omi[['Compr_min', 'Compr_max']].mean(axis=1)
residential = omi[omi['Descr_Tipologia'].astype('string').str.contains('abitazion|villa', case=False, na=False)].copy()
semester_panel = (residential.dropna(subset=['municipality_code', 'price_m2'])
    .groupby(['year', 'semester', 'municipality_code'], as_index=False)
    .agg(price_m2=('price_m2', 'median'), quotation_obs=('price_m2', 'size'), region=('Regione', 'first')))
annual_price = (semester_panel.groupby(['year', 'municipality_code'], as_index=False)
    .agg(price_m2=('price_m2', 'mean'), quotation_obs=('quotation_obs', 'sum'), region=('region', 'first'), semesters_available=('semester', 'nunique')))

transaction_parts = []
for folder in sorted(p for p in TRANSACTIONS_DIR.iterdir() if p.is_dir() and p.name.isdigit()):
    year = int(folder.name)
    files = [p for p in folder.iterdir() if p.is_file()]
    lista_path = next(p for p in files if 'lista-com' in p.name.lower())
    res_path = next(p for p in files if 'valori-res' in p.name.lower())
    lista = pd.read_csv(lista_path, sep=';', decimal=',')
    res = pd.read_csv(res_path, sep=';', decimal=',')
    lista.columns = [str(c).strip() for c in lista.columns]
    res.columns = [str(c).strip() for c in res.columns]
    lista_code = next(c for c in lista.columns if re.search(r'codcom$', c, re.I))
    res_code = next(c for c in res.columns if re.search(r'codcom$', c, re.I))
    ntn_candidates = [c for c in res.columns if re.fullmatch(r'NTN_?'+str(year), c, re.I)]
    if not ntn_candidates:
        ntn_candidates = [c for c in res.columns if re.match(r'NTN', c, re.I) and 'mq' not in c.lower()]
    ntn_col = ntn_candidates[0]
    geo_cols = [c for c in [lista_code, 'Comune', 'Provincia', 'Regione'] if c in lista.columns]
    geo = lista[geo_cols].rename(columns={lista_code: 'municipality_code'}).copy()
    vals = res[[res_code, ntn_col]].rename(columns={res_code: 'municipality_code', ntn_col: 'ntn'}).copy()
    vals['ntn'] = pd.to_numeric(vals['ntn'], errors='coerce')
    if vals['municipality_code'].duplicated().any():
        raise ValueError(f'{year}: duplicate NTN municipality key')
    transaction_parts.append(geo.merge(vals, on='municipality_code', how='left', validate='one_to_one').assign(year=year))
transactions = pd.concat(transaction_parts, ignore_index=True)

population_parts = []
for folder in sorted(p for p in POPULATION_DIR.iterdir() if p.is_dir() and p.name.isdigit()):
    year = int(folder.name)
    path = next(folder.glob('*_Comuni.csv'))
    df = pd.read_csv(path, sep=';', encoding='utf-8-sig', usecols=['Codice comune', 'Età', 'Totale'])
    df.columns = ['municipality_code', 'age', 'population']
    df['municipality_code'] = df['municipality_code'].astype('string').str.strip()
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df['population'] = pd.to_numeric(df['population'], errors='coerce')
    totals = df.loc[df['age'].eq(999), ['municipality_code', 'population']].copy()
    population_parts.append(totals.assign(year=year))
population = pd.concat(population_parts, ignore_index=True)

market = annual_price.merge(transactions[['year', 'municipality_code', 'ntn']], on=['year', 'municipality_code'], how='left', validate='many_to_one')
market = market.merge(population, on=['year', 'municipality_code'], how='left', validate='many_to_one')
market = market.sort_values(['municipality_code', 'year']).reset_index(drop=True)
assert not market.duplicated(['year', 'municipality_code']).any()
print(f'Annual municipality panel: {len(market):,} rows')
display(market.head())

## 3. Feature engineering without temporal leakage

All explanatory variables are lagged by one year. Therefore, the model for year *t* only uses information observed at or before year *t-1*. This is the key safeguard against look-ahead bias.

In [ ]:
g = market.groupby('municipality_code', group_keys=False)
market['price_growth'] = g['price_m2'].pct_change()
market['ntn_growth'] = g['ntn'].pct_change()
market['population_growth'] = g['population'].pct_change()
market['price_growth_lag1'] = g['price_growth'].shift(1)
market['ntn_growth_lag1'] = g['ntn_growth'].shift(1)
market['population_growth_lag1'] = g['population_growth'].shift(1)
market['price_level_lag1'] = g['price_m2'].shift(1)
market['ntn_level_lag1'] = g['ntn'].shift(1)
market['population_level_lag1'] = g['population'].shift(1)
market['target_price_growth'] = g['price_growth'].shift(-1)

model_data = market.dropna(subset=['target_price_growth', 'price_growth_lag1', 'ntn_growth_lag1', 'population_growth_lag1']).copy()
print(f'Model observations: {len(model_data):,}')
display(model_data[['year', 'municipality_code', 'price_growth_lag1', 'ntn_growth_lag1', 'population_growth_lag1', 'target_price_growth']].head())

## 4. Exploratory target analysis

Before fitting models, inspect the distribution of the forecasting target and its evolution over time.

In [ ]:
annual_target = model_data.groupby('year')['target_price_growth'].agg(['count', 'mean', 'median', 'std']).reset_index()
display(annual_target)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(annual_target['year'], annual_target['mean'], marker='o', label='Mean target growth')
ax.plot(annual_target['year'], annual_target['median'], marker='o', label='Median target growth')
ax.axhline(0, linestyle='--', linewidth=1)
ax.set_title('Next-Year Municipality Price Growth')
ax.set_xlabel('Forecast year')
ax.set_ylabel('Price growth')
ax.legend()
fig.tight_layout()
plt.show()

## 5. Chronological train/test split

The final 20% of available years are held out as the test period. The split is performed by **year**, not by row, so municipalities from the same calendar year cannot appear in both train and test samples.

In [ ]:
years = np.sort(model_data['year'].unique())
n_test_years = max(1, int(np.ceil(len(years) * 0.20)))
train_years = years[:-n_test_years]
test_years = years[-n_test_years:]
train = model_data[model_data['year'].isin(train_years)].copy()
test = model_data[model_data['year'].isin(test_years)].copy()
print('Training years:', train_years)
print('Test years:', test_years)
print(f'Train rows: {len(train):,} | Test rows: {len(test):,}')

## 6. Baseline and Ridge forecasting models

The baseline predicts the training-set mean. Ridge regression provides a transparent regularized benchmark and is intentionally preferred before introducing more complex tree-based models.

The pipeline imputes missing values and standardizes predictors using **training data only** through scikit-learn's `Pipeline`.

In [ ]:
features = ['price_growth_lag1', 'ntn_growth_lag1', 'population_growth_lag1', 'price_level_lag1', 'ntn_level_lag1', 'population_level_lag1']
target = 'target_price_growth'
X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

ridge = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=10.0))
])
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)

def metrics(y_true, pred):
    return {
        'MAE': mean_absolute_error(y_true, pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, pred)),
        'R2': r2_score(y_true, pred),
    }

results = pd.DataFrame([metrics(y_test, baseline_pred), metrics(y_test, ridge_pred)], index=['Baseline mean', 'Ridge'])
display(results)

## 7. Out-of-sample prediction diagnostics

The key comparison is against the baseline. A model is useful only if it improves predictive accuracy on observations it did not see during training.

In [ ]:
predictions = test[['year', 'municipality_code', 'region', target]].copy()
predictions['baseline_pred'] = baseline_pred
predictions['ridge_pred'] = ridge_pred
predictions['ridge_error'] = predictions[target] - predictions['ridge_pred']
display(predictions.head(20))

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y_test, ridge_pred, alpha=0.35)
limits = [min(y_test.min(), ridge_pred.min()), max(y_test.max(), ridge_pred.max())]
ax.plot(limits, limits, linestyle='--', linewidth=1)
ax.set_title('Out-of-Sample: Actual vs Ridge Forecast')
ax.set_xlabel('Actual next-year price growth')
ax.set_ylabel('Predicted next-year price growth')
fig.tight_layout()
plt.show()

## 8. Feature importance / coefficient interpretation

Ridge coefficients are standardized, so their absolute magnitude provides a directional indication of relative predictive contribution. They should not be interpreted as causal effects.

In [ ]:
coef = pd.Series(ridge.named_steps['model'].coef_, index=features).sort_values()
display(coef.to_frame('standardized_coefficient'))

fig, ax = plt.subplots(figsize=(9, 5))
coef.plot(kind='barh', ax=ax)
ax.axvline(0, linestyle='--', linewidth=1)
ax.set_title('Ridge Coefficients')
ax.set_xlabel('Standardized coefficient')
fig.tight_layout()
plt.show()

## 9. Error analysis by forecast year

Aggregate errors by year to identify periods in which the model performs particularly well or poorly.

In [ ]:
year_errors = (predictions.groupby('year')
    .apply(lambda x: pd.Series({
        'MAE': mean_absolute_error(x[target], x['ridge_pred']),
        'RMSE': np.sqrt(mean_squared_error(x[target], x['ridge_pred'])),
        'mean_actual': x[target].mean(),
        'mean_predicted': x['ridge_pred'].mean(),
    }), include_groups=False)
    .reset_index())
display(year_errors)

## 10. Conclusions and limitations

### What this notebook establishes
- A reproducible chronological forecasting setup.
- Explicit lagged features and a leakage-resistant pipeline.
- A transparent baseline versus regularized regression benchmark.
- Out-of-sample evaluation with MAE, RMSE and R².

### What it does not establish
- Predictive performance is not causal inference.
- OMI quotation estimates are not transaction prices.
- Population and NTN are aggregate indicators and may omit relevant supply, income, mortgage-rate and macroeconomic variables.
- A single chronological holdout can be sensitive to the chosen test period.
- Municipality observations are not necessarily independent; spatial dependence may remain.

### Next step
The next iteration can introduce rolling-origin cross-validation, stronger regularized models and tree-based algorithms, followed by a strict comparison against the Ridge benchmark.